# Compare SDXL Turbo and SDXL Base on a prompt

Enter a prompt and click **Submit both SLURM jobs**. The notebook submits one job for each model from the cluster login node. Click **Refresh status / show images** to view whichever batches have finished.

Open this notebook in Jupyter on the cluster login node, where `sbatch` is available. Generation runs on SLURM GPU nodes. Same numeric seeds do not make images directly paired because the models use different resolutions and sampling behavior.

In [ ]:
from pathlib import Path
import sys
import json
import subprocess
import shutil
from datetime import datetime
from IPython.display import display, clear_output, Image as DisplayImage
import ipywidgets as widgets

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "generation/generation_utils.py").exists())
SLURM_SCRIPT = ROOT / "generation/generate_batch.slurm"
RUN_ROOT = ROOT / "run_outputs"
CONFIG_DIR = RUN_ROOT / "configs"
IMAGE_ROOT = RUN_ROOT / "images"
LOG_DIR = RUN_ROOT / "logs"
for folder in (CONFIG_DIR, IMAGE_ROOT, LOG_DIR):
    folder.mkdir(parents=True, exist_ok=True)

if shutil.which("sbatch") is None:
    raise RuntimeError("sbatch is not available. Open this notebook in Jupyter on the cluster login node.")

jobs = {}

def submit_job(model_name, prompt_text, image_count, first_seed, inference_steps, guidance_scale, run_name):
    config = {
        "model_name": model_name,
        "prompt": prompt_text,
        "count": int(image_count),
        "seed": int(first_seed),
        "steps": int(inference_steps),
        "guidance": float(guidance_scale),
        "output_root": str(IMAGE_ROOT),
        "run_name": run_name,
    }
    config_path = CONFIG_DIR / f"{run_name}.json"
    config_path.write_text(json.dumps(config, indent=2) + "\n")
    result = subprocess.run(
        ["sbatch", "--parsable", str(SLURM_SCRIPT), str(config_path)],
        cwd=ROOT, text=True, capture_output=True, check=True,
    )
    job_id = result.stdout.strip().split(";")[0]
    jobs[job_id] = IMAGE_ROOT / run_name
    print(f"Submitted {model_name} job {job_id}; output: {jobs[job_id]}")
    print(f"SLURM log: {LOG_DIR / f'sdxl_batch-{job_id}.out'}")
    return job_id

def refresh_jobs(_):
    with output:
        clear_output(wait=True)
        if not jobs:
            print("Submit a generation job first.")
            return
        for job_id, output_dir in jobs.items():
            queue = subprocess.run(["squeue", "-h", "-j", job_id, "-o", "%T"],
                                   text=True, capture_output=True)
            state = queue.stdout.strip()
            if not state:
                accounting = subprocess.run(["sacct", "-n", "-X", "-j", job_id, "--format=State"],
                                            text=True, capture_output=True)
                state = next((line.strip() for line in accounting.stdout.splitlines() if line.strip()), "No status returned")
            print(f"Job {job_id}: {state}")
            images = sorted(output_dir.glob("image_*.png"))
            if images:
                display(*[DisplayImage(filename=str(p)) for p in images])
            elif state.startswith("FAILED") or state.startswith("CANCELLED") or state.startswith("OUT_OF_MEMORY"):
                log = LOG_DIR / f"sdxl_batch-{job_id}.out"
                if log.exists():
                    print("Last log lines:\n" + "\n".join(log.read_text(errors="replace").splitlines()[-30:]))


## Configure and submit

The notebook uses the model-specific default profiles: Turbo is one step at 512 × 512; Base is 30 steps at 1024 × 1024. Each image is generated separately to reduce GPU memory use. Both jobs write their configs, logs, and images under `run_outputs/`.

In [ ]:
prompt = widgets.Textarea(value="A photo of a flower in a garden", description="Prompt:", layout=widgets.Layout(width="90%", height="90px"))
count = widgets.IntSlider(value=2, min=1, max=6, step=1, description="Images/model:")
seed = widgets.IntText(value=0, description="First seed:")
submit_button = widgets.Button(description="Submit both SLURM jobs", button_style="primary")
refresh_button = widgets.Button(description="Refresh status / show images")
output = widgets.Output()

def on_compare_submit(_):
    with output:
        clear_output(wait=True)
        if not prompt.value.strip():
            print("Enter a prompt first.")
            return
        stamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
        for model_name, steps, guidance in [("SDXL Turbo", 1, 0.0), ("SDXL Base", 30, 5.0)]:
            try:
                submit_job(model_name, prompt.value, count.value, seed.value,
                           steps, guidance, f"compare_{stamp}_{model_name.lower().replace(' ', '_')}")
            except Exception as exc:
                print(f"Could not submit {model_name}: {type(exc).__name__}: {exc}")
        print("Both model runs use SLURM GPU jobs. Click Refresh status / show images later.")

submit_button.on_click(on_compare_submit)
refresh_button.on_click(refresh_jobs)
display(prompt, count, seed, submit_button, refresh_button, output)


The output folders include a `settings.json` file for each completed run. If a job is still queued or running, use the refresh button again later, or check it with `squeue -u $USER`.